## Cell 1 — Install Dependencies


# Financial LLM Finetuning Notebook
**Models**: SmolLM2-360M-Instruct | Qwen3-0.6B | Gemma-3-270m-it  
**Hardware**: Kaggle T4 GPU (16 GB VRAM)  
**Method**: QLoRA (4-bit NF4) + LoRA (r=16, alpha=32)  
**Benchmarks**: A) Merged finetuning → per-dataset eval | B) Individual dataset finetuning  
**Metrics**: BLEU, ROUGE-1/2/L, Token-F1, Sentiment Macro-F1



In [ ]:
# !pip install -q transformers
# !pip install -q datasets
!pip install -q peft
!pip install -q trl
!pip install -q bitsandbytes>=0.46.1
# !pip install -q accelerate
# !pip install -q sentencepiece
# !pip install -q protobuf
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q nltk
!pip install -q sacrebleu
# !pip install -q scikit-learn
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q tensorboard
# !pip install -q tqdm
# !pip install -q torch


## Cell 2 — All Imports


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import re
import json
import math
import copy
import random
import logging
import warnings
import itertools
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any

import nltk
import numpy as np
import pandas as pd
import torch
import evaluate
from tqdm.auto import tqdm

from datasets import (
    load_dataset,
    Dataset,
    DatasetDict,
    concatenate_datasets,
    interleave_datasets,
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback,
    TrainerState,
    TrainerControl,
    DataCollatorForSeq2Seq,
    GenerationConfig,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

from trl import SFTTrainer, SFTConfig

from sklearn.metrics import f1_score, classification_report
from accelerate import Accelerator
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"]=user_secrets.get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
accelerator = Accelerator(mixed_precision="no")

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("FinanceFT")

torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Device: {DEVICE}")
if DEVICE == "cuda":
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
DEVICE

## Cell 3 — Master Configuration


In [ ]:

# ─── Dataset Config ───────────────────────────────────────────────────────────
# size_used: number of samples to draw from the full dataset for finetuning.
#            Set to None to use the entire dataset.
#            80% → train, 20% → test split applied after sampling.
DATASET_CONFIG = {
    "finder": {
        "hf_path": "Linq-AI-Research/FinDER",
        "hf_split": "train",
        "size_used": 1500,
        "task_group": "qa",
        "enabled": True,
    },
    # "convfinqa": { # unseen testing 
    #     "hf_path": "ChanceFocus/flare-convfinqa", # unseen
    #     "hf_split": "train",
    #     "size_used": 2000,
    #     "task_group": "qa",
    #     "enabled": True,
    # },
    "tatqa": {
        "hf_path": "next-tat/TAT-QA",
        "hf_split": "train",
        "size_used": 1500,
        "task_group": "qa",
        "enabled": True,
    },
    "fiqaqa": {
        "hf_path": "FinGPT/fingpt-fiqa_qa",
        "hf_split": "train",
        "size_used": 1500,
        "task_group": "qa",
        "enabled": True,
    },
    
    "fiqasa": {
        "hf_path": "ChanceFocus/flare-fiqasa",
        "hf_split": "train",
        "size_used": 3750,
        "task_group": "sentiment",
        "enabled": True,
    },
    "finnews": {
        # Placeholder — upload dataset.csv to Kaggle as a dataset
        # and set the path below before running
        "local_path": "/kaggle/input/datasets/sayelabualigah/high-quality-financial-news-dataset-for-nlp-tasks/dataset.csv",
        "size_used": 3750,
        "task_group": "summarization",
        "enabled": True,
    },
    "finNer": {
        "hf_path": "Josephgflowers/Financial-NER-NLP",
        "hf_split": "train",
        "size_used": 2000,
        "task_group": "ner",
        "enabled": True,
    },
}

# ─── Model Config ─────────────────────────────────────────────────────────────
MODEL_CONFIG = {
    "gemma3": {
        "model_id": "google/gemma-3-270m-it",
        "enabled": True,
        "chat_template": "gemma",
        "max_seq_length": 512,
    },
    "smollm2": {
        "model_id": "HuggingFaceTB/SmolLM2-360M-Instruct",
        "enabled": False,
        "chat_template": "chatml",          # uses <|im_start|> format
        "max_seq_length": 512,
    },
    "qwen3": {
        "model_id": "Qwen/Qwen3-0.6B",
        "enabled": False,
        "chat_template": "chatml",
        "enable_thinking": False,           # /think token disabled
        "max_seq_length": 512,
    },

}

# ─── LoRA Config (identical for all models — equal comparison) ────────────────
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM,
    "target_modules": "all-linear",        # will be resolved per model
}

# ─── Training Config ──────────────────────────────────────────────────────────
TRAINING_CONFIG = {
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,      # effective batch = 16
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "fp16": False,
    "bf16": False,
    "logging_steps": 25,
    "eval_steps": 100,
    "save_steps": 100,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "report_to": "tensorboard",
    "dataloader_num_workers": 2,
    "optim": "paged_adamw_8bit",
    "neftune_noise_alpha": 5,
}

# ─── QLoRA quantisation ───────────────────────────────────────────────────────
BNBCONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE_OUTPUT_DIR = "/kaggle/working/checkpoints"
RESULTS_DIR     = "/kaggle/working/results"
LOGS_DIR        = "/kaggle/working/logs"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

TRAIN_TEST_SPLIT = 0.8   # 80 / 20

## Cell 4 — Dataset Loaders & Formatters


In [ ]:

# ─── Prompt template ──────────────────────────────────────────────────────────
def make_prompt(instruction: str, context: str, input_text: str, output: str = "") -> str:
    """Wrap a sample in the unified generation format."""
    prompt = (
        f"<instruction>{instruction.strip()}</instruction>\n"
        f"<context>{context.strip()}</context>\n"
        f"<input>{input_text.strip()}</input>\n"
        f"<output>{output.strip()}</output>"
    )
    return prompt


def make_inference_prompt(instruction: str, context: str, input_text: str) -> str:
    """Inference-time prompt — output tag left open for generation."""
    return (
        f"<instruction>{instruction.strip()}</instruction>\n"
        f"<context>{context.strip()}</context>\n"
        f"<input>{input_text.strip()}</input>\n"
        f"<output>"
    )


# ─── FinDER ───────────────────────────────────────────────────────────────────
def format_finder(sample: dict) -> Optional[dict]:
    question = str(sample.get("text", "")).strip()
    answer   = str(sample.get("answer", "")).strip()
    category = str(sample.get("category", "general")).strip()
    if not question or not answer:
        return None
    return {
        "text": make_prompt(
            instruction="Answer the following financial question accurately and concisely.",
            context=f"Category: {category}",
            input_text=question,
            output=answer,
        ),
        "task": "qa",
        "dataset": "finder",
        "reference": answer,
    }


# ─── ConvFinQA ────────────────────────────────────────────────────────────────
def format_convfinqa(sample: dict) -> Optional[dict]:
    query  = str(sample.get("query", "")).strip()
    answer = str(sample.get("answer", "")).strip()
    if not query or not answer:
        return None
    turn = sample.get("turn", 0)
    ctx  = f"Conversation turn: {turn}" if turn is not None else "Conversational financial QA"
    return {
        "text": make_prompt(
            instruction="Answer this conversational financial question based on the provided context.",
            context=ctx,
            input_text=query,
            output=answer,
        ),
        "task": "qa",
        "dataset": "convfinqa",
        "reference": answer,
    }


# ─── TAT-QA ───────────────────────────────────────────────────────────────────
def _tatqa_table_to_text(table: Any) -> str:
    """Convert a nested list table to a pipe-delimited string."""
    if not table or not isinstance(table, list):
        return ""
    rows = []
    for row in table:
        if isinstance(row, list):
            rows.append(" | ".join(str(c).strip() for c in row))
    return "\n".join(rows)


def _tatqa_paragraphs_to_text(paragraphs: Any) -> str:
    if not paragraphs or not isinstance(paragraphs, list):
        return ""
    texts = []
    for p in sorted(paragraphs, key=lambda x: x.get("order", 0) if isinstance(x, dict) else 0):
        if isinstance(p, dict):
            texts.append(str(p.get("text", "")).strip())
    return " ".join(texts)


def format_tatqa(sample: dict) -> List[dict]:
    """One TAT-QA document → one sample per question."""
    table      = sample.get("table", {})
    paragraphs = sample.get("paragraphs", [])
    questions  = sample.get("questions", [])

    if isinstance(table, dict):
        table_data = table.get("table", table)
    else:
        table_data = table

    table_text = _tatqa_table_to_text(table_data)
    para_text  = _tatqa_paragraphs_to_text(paragraphs)
    context    = f"Table:\n{table_text}\n\nText:\n{para_text}".strip()

    results = []
    for q in (questions if isinstance(questions, list) else []):
        if not isinstance(q, dict):
            continue
        question   = str(q.get("question", "")).strip()
        raw_answer = q.get("answer", "")
        derivation = str(q.get("derivation", "")).strip()
        scale      = str(q.get("scale", "")).strip()

        if isinstance(raw_answer, list):
            answer_str = ", ".join(str(a) for a in raw_answer)
        else:
            answer_str = str(raw_answer).strip()

        if scale and scale not in ("", "None"):
            answer_str = f"{answer_str} ({scale})"
        if derivation:
            answer_str = f"{answer_str}\nDerivation: {derivation}"

        if not question or not answer_str:
            continue

        results.append({
            "text": make_prompt(
                instruction=(
                    "Answer the financial question using the provided table and text passage. "
                    "Show your derivation if numerical calculation is required."
                ),
                context=context,
                input_text=question,
                output=answer_str,
            ),
            "task": "qa",
            "dataset": "tatqa",
            "reference": answer_str,
        })
    return results


# ─── FiQA-QA ──────────────────────────────────────────────────────────────────
def format_fiqaqa(sample: dict) -> Optional[dict]:
    instruction_field = str(sample.get("instruction", "")).strip()
    input_field       = str(sample.get("input", "")).strip()
    output_field      = str(sample.get("output", "")).strip()
    if not input_field or not output_field:
        return None
    instr = instruction_field if instruction_field else "Answer the financial opinion question."
    return {
        "text": make_prompt(
            instruction=instr,
            context="Financial opinion mining and question answering.",
            input_text=input_field,
            output=output_field,
        ),
        "task": "qa",
        "dataset": "fiqaqa",
        "reference": output_field,
    }


# ─── PACIFIC ──────────────────────────────────────────────────────────────────
def format_pacific(sample: dict) -> List[dict]:
    """PACIFIC mirrors TAT-QA structure — table + paragraphs + questions."""
    table      = sample.get("table", {})
    paragraphs = sample.get("paragraphs", [])
    questions  = sample.get("questions", [])

    if isinstance(table, dict):
        table_data = table.get("table", table)
    else:
        table_data = table

    table_text = _tatqa_table_to_text(table_data)
    para_text  = _tatqa_paragraphs_to_text(paragraphs)
    context    = f"Table:\n{table_text}\n\nText:\n{para_text}".strip()

    results = []
    for q in (questions if isinstance(questions, list) else []):
        if not isinstance(q, dict):
            continue
        question   = str(q.get("question", "")).strip()
        raw_answer = q.get("answer", "")
        derivation = str(q.get("derivation", "")).strip()

        if isinstance(raw_answer, list):
            answer_str = ", ".join(str(a) for a in raw_answer)
        else:
            answer_str = str(raw_answer).strip()

        if derivation:
            answer_str = f"{answer_str}\nDerivation: {derivation}"

        if not question or not answer_str:
            continue

        results.append({
            "text": make_prompt(
                instruction=(
                    "Answer the financial question proactively using the table and passage. "
                    "Show derivation if numerical reasoning is involved."
                ),
                context=context,
                input_text=question,
                output=answer_str,
            ),
            "task": "qa",
            "dataset": "pacific",
            "reference": answer_str,
        })
    return results


# ─── Financial PhraseBank ─────────────────────────────────────────────────────
_SENTIMENT_MAP = {0: "negative", 1: "neutral", 2: "positive"}

def format_phrasebank(sample: dict) -> Optional[dict]:
    sentence = str(sample.get("sentence", "")).strip()
    label    = sample.get("label", None)
    if not sentence or label is None:
        return None
    sentiment = _SENTIMENT_MAP.get(int(label), "neutral")
    return {
        "text": make_prompt(
            instruction=(
                "Classify the sentiment of the following financial sentence. "
                "Respond with exactly one word: positive, negative, or neutral."
            ),
            context="Financial sentiment analysis.",
            input_text=sentence,
            output=sentiment,
        ),
        "task": "sentiment",
        "dataset": "phrasebank",
        "reference": sentiment,
    }


# ─── FiQA-SA ──────────────────────────────────────────────────────────────────
def format_fiqasa(sample: dict) -> Optional[dict]:
    text   = str(sample.get("text", "")).strip()
    answer = str(sample.get("answer", "")).strip().lower()
    query  = str(sample.get("query", "")).strip()
    if not text or not answer:
        return None
    if answer not in ("positive", "negative", "neutral"):
        return None
    return {
        "text": make_prompt(
            instruction=(
                "Classify the sentiment of the following financial text snippet. "
                "Respond with exactly one word: positive, negative, or neutral."
            ),
            context=query if query else "Financial sentiment classification.",
            input_text=text,
            output=answer,
        ),
        "task": "sentiment",
        "dataset": "fiqasa",
        "reference": answer,
    }


# ─── High-Quality Financial News ─────────────────────────────────────────────
def format_finnews(sample: dict) -> Optional[dict]:
    content = str(sample.get("Content", "")).strip()
    summary = str(sample.get("CompactedSummary", "") or sample.get("DetailedSummary", "")).strip()
    subject = str(sample.get("Subject", "")).strip()
    if not content or not summary:
        return None
    ctx = f"Company/Entity: {subject}" if subject else "Financial news summarization."
    return {
        "text": make_prompt(
            instruction="Summarize the following financial news article concisely and accurately.",
            context=ctx,
            input_text=content[:1500],   # guard very long articles
            output=summary,
        ),
        "task": "summarization",
        "dataset": "finnews",
        "reference": summary,
    }


# ─── Financial NER ────────────────────────────────────────────────────────────
def format_finNer(sample: dict) -> Optional[dict]:
    system    = str(sample.get("system", "")).strip()
    user_text = str(sample.get("user", "")).strip()
    assistant = str(sample.get("assistant", "")).strip()
    if not user_text or not assistant:
        return None
    ctx = system if system else "Financial named entity recognition and extraction."
    return {
        "text": make_prompt(
            instruction=(
                "Identify and extract all financial named entities (XBRL tags) "
                "from the following text."
            ),
            context=ctx,
            input_text=user_text,
            output=assistant,
        ),
        "task": "ner",
        "dataset": "finNer",
        "reference": assistant,
    }


## Cell 5 — Dataset Loading Pipeline


In [ ]:

def _sample_dataset(ds: Dataset, size_used: Optional[int], seed: int = 42) -> Dataset:
    """Draw `size_used` samples; if None, return full dataset."""
    if size_used is None or size_used >= len(ds):
        return ds
    return ds.shuffle(seed=seed).select(range(size_used))


def _apply_formatter_flat(ds: Dataset, formatter) -> List[dict]:
    """Apply a formatter that returns Optional[dict] (one sample → one output)."""
    results = []
    for sample in tqdm(ds, desc="Formatting", leave=False):
        out = formatter(sample)
        if out is not None:
            results.append(out)
    return results


def _apply_formatter_expand(ds: Dataset, formatter) -> List[dict]:
    """Apply a formatter that returns List[dict] (one sample → many outputs)."""
    results = []
    for sample in tqdm(ds, desc="Formatting", leave=False):
        results.extend(formatter(sample))
    return results


def load_and_format_all(dataset_config: dict) -> Dict[str, List[dict]]:
    """
    Load every enabled dataset, apply size limit, format to unified schema.
    Returns dict[dataset_key → list of formatted samples].
    """
    formatted = {}

    for key, cfg in dataset_config.items():
        if not cfg.get("enabled", True):
            logger.info(f"Skipping {key} (disabled)")
            continue

        logger.info(f"Loading dataset: {key}")

        try:
            # ── Load raw ────────────────────────────────────────────────────
            if "hf_path" in cfg:
                raw = load_dataset(cfg["hf_path"], split=cfg.get("hf_split", "train"),)
            elif "local_path" in cfg:
                p = cfg["local_path"]
                if not os.path.exists(p):
                    logger.warning(f"  Placeholder path not found: {p}. Skipping {key}.")
                    continue
                if p.endswith(".json"):
                    with open(p) as f:
                        data = json.load(f)
                    raw = Dataset.from_list(data if isinstance(data, list) else [data])
                elif p.endswith(".csv"):
                    df  = pd.read_csv(p)
                    raw = Dataset.from_pandas(df)
                else:
                    logger.warning(f"  Unsupported local format for {key}. Skipping.")
                    continue
            else:
                logger.warning(f"  No path for {key}. Skipping.")
                continue

            logger.info(f"  Raw size: {len(raw)}")

            # ── Sample ──────────────────────────────────────────────────────
            raw = _sample_dataset(raw, cfg.get("size_used"))
            logger.info(f"  After sampling: {len(raw)}")

            # ── Format ──────────────────────────────────────────────────────
            if key == "finder":
                samples = _apply_formatter_flat(raw, format_finder)
            elif key == "convfinqa":
                samples = _apply_formatter_flat(raw, format_convfinqa)
            elif key == "tatqa":
                samples = _apply_formatter_expand(raw, format_tatqa)
            elif key == "fiqaqa":
                samples = _apply_formatter_flat(raw, format_fiqaqa)
            elif key == "pacific":
                samples = _apply_formatter_expand(raw, format_pacific)
            elif key == "phrasebank":
                samples = _apply_formatter_flat(raw, format_phrasebank)
            elif key == "fiqasa":
                samples = _apply_formatter_flat(raw, format_fiqasa)
            elif key == "finnews":
                samples = _apply_formatter_flat(raw, format_finnews)
            elif key == "finNer":
                samples = _apply_formatter_flat(raw, format_finNer)
            else:
                logger.warning(f"  Unknown key {key}. Skipping.")
                continue

            logger.info(f"  Formatted samples: {len(samples)}")
            formatted[key] = samples

        except Exception as e:
            logger.error(f"  Failed to load/format {key}: {e}")
            continue

    return formatted


def split_dataset(samples: List[dict], train_ratio: float = TRAIN_TEST_SPLIT, seed: int = 42
                  ) -> Tuple[List[dict], List[dict]]:
    """80/20 deterministic split."""
    random.seed(seed)
    shuffled = samples.copy()
    random.shuffle(shuffled)
    cut = int(len(shuffled) * train_ratio)
    return shuffled[:cut], shuffled[cut:]


def build_dataset_splits(formatted: Dict[str, List[dict]]) -> Dict[str, Dict[str, List[dict]]]:
    """Returns {dataset_key: {'train': [...], 'test': [...]}}"""
    splits = {}
    for key, samples in formatted.items():
        train, test = split_dataset(samples)
        splits[key] = {"train": train, "test": test}
        logger.info(f"  {key}: {len(train)} train | {len(test)} test")
    return splits


# ─── Run loaders ─────────────────────────────────────────────────────────────
logger.info("=" * 60)
logger.info("Loading and formatting all datasets")
logger.info("=" * 60)
ALL_FORMATTED  = load_and_format_all(DATASET_CONFIG)
DATASET_SPLITS = build_dataset_splits(ALL_FORMATTED)


## Cell 6 — Model Factory & LoRA Setup


## Cell 7 — Checkpoint & Logging Callbacks


In [ ]:
def get_target_modules(model_key: str) -> List[str]:
    """
    Return LoRA target module names per architecture.
    All models → attention projections + gate/up/down proj for equal footing.
    """
    if model_key == "smollm2":
        return ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
    elif model_key == "qwen3":
        return ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
    elif model_key == "gemma3":
        return ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
    return ["q_proj", "v_proj"]


def load_model_and_tokenizer(model_key: str, model_cfg: dict):
    """Load quantised model + tokenizer, attach LoRA adapter."""
    model_id = model_cfg["model_id"]
    logger.info(f"Loading model: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        padding_side="right",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=BNBCONFIG,
        device_map={"": 0},
        dtype=torch.float32,
        attn_implementation="eager"
    )

    model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"],
        target_modules=get_target_modules(model_key),
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    return model, tokenizer


def apply_chat_template(model_key: str, tokenizer, sample_text: str) -> str:
    """
    Wrap the unified prompt in the model-specific chat format
    so the model sees the correct special tokens.
    """
    # Extract output (target) from prompt
    if "<output>" in sample_text and "</output>" in sample_text:
        prefix = sample_text[:sample_text.rfind("<output>") + len("<output>")]
        output = sample_text.split("<output>")[-1].replace("</output>", "").strip()
    else:
        prefix = sample_text
        output = ""

    if model_key in ("smollm2", "qwen3"):
        # ChatML
        full = (
            f"<|im_start|>user\n{prefix}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )
    elif model_key == "gemma3":
        full = (
            f"<start_of_turn>user\n{prefix}<end_of_turn>\n"
            f"<start_of_turn>model\n{output}<end_of_turn>"
        )
    else:
        full = sample_text

    return full


def tokenize_dataset(model_key: str, tokenizer, samples: List[dict],
                      max_length: int) -> Dataset:
    """
    Convert list of formatted samples to a tokenised HF Dataset.
    """
    texts = [apply_chat_template(model_key, tokenizer, s["text"]) for s in samples]
    ds    = Dataset.from_dict({"text": texts})
    return ds

In [ ]:

class FinanceTrainingCallback(TrainerCallback):
    """
    Custom callback:
    - Logs train/eval loss to a JSONL file per run.
    - Prints a summary line every `log_every` steps.
    - Saves best checkpoint path to a text file.
    """

    def __init__(self, log_path: str, log_every: int = 25):
        self.log_path  = log_path
        self.log_every = log_every
        self.best_eval = float("inf")
        self._fh = open(log_path, "w")

    def _write(self, record: dict):
        self._fh.write(json.dumps(record) + "\n")
        self._fh.flush()

    def on_log(self, args, state: TrainerState, control: TrainerControl, logs=None, **kwargs):
        if logs is None:
            return
        record = {"step": state.global_step, "epoch": state.epoch, **logs}
        self._write(record)
        if state.global_step % self.log_every == 0:
            parts = [f"step={state.global_step}", f"epoch={state.epoch:.2f}"]
            for k in ("loss", "eval_loss", "learning_rate"):
                if k in logs:
                    parts.append(f"{k}={logs[k]:.4f}")
            logger.info("  [Train] " + " | ".join(parts))

    def on_evaluate(self, args, state: TrainerState, control: TrainerControl, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            el = metrics["eval_loss"]
            if el < self.best_eval:
                self.best_eval = el
                logger.info(f"  ✓ New best eval_loss={el:.4f} at step {state.global_step}")

    def on_train_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        self._fh.close()
        logger.info(f"  Training log saved to {self.log_path}")


## Cell 8 — Training Function (Generic)


In [ ]:

def run_training(
    model_key: str,
    run_name: str,
    train_samples: List[dict],
    eval_samples: List[dict],
    output_dir: str,
    train:bool,
):
    """
    Full training run for a given model + dataset split.
    Handles model loading, tokenisation, SFTTrainer, checkpointing.
    Returns path to best checkpoint.
    """
    model_cfg = MODEL_CONFIG[model_key]
    if not model_cfg["enabled"]:
        logger.info(f"Model {model_key} disabled. Skipping.")
        return None

    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(LOGS_DIR, f"{run_name}.jsonl")

    logger.info("=" * 60)
    logger.info(f"RUN: {run_name}")
    logger.info(f"  Model   : {model_cfg['model_id']}")
    logger.info(f"  Train   : {len(train_samples)} | Eval: {len(eval_samples)}")
    logger.info("=" * 60)

    model, tokenizer = load_model_and_tokenizer(model_key, model_cfg)

    train_ds = tokenize_dataset(model_key, tokenizer, train_samples,
                                model_cfg["max_seq_length"])
    eval_ds  = tokenize_dataset(model_key, tokenizer, eval_samples,
                                model_cfg["max_seq_length"])

    training_args = SFTConfig(
        output_dir=output_dir,
        run_name=run_name,
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
        per_device_eval_batch_size=TRAINING_CONFIG["per_device_eval_batch_size"],
        gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
        warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
        weight_decay=TRAINING_CONFIG["weight_decay"],
        fp16=TRAINING_CONFIG["fp16"],
        bf16=TRAINING_CONFIG["bf16"],
        logging_steps=TRAINING_CONFIG["logging_steps"],
        eval_steps=TRAINING_CONFIG["eval_steps"],
        save_steps=TRAINING_CONFIG["save_steps"],
        save_total_limit=TRAINING_CONFIG["save_total_limit"],
        load_best_model_at_end=TRAINING_CONFIG["load_best_model_at_end"],
        metric_for_best_model=TRAINING_CONFIG["metric_for_best_model"],
        greater_is_better=TRAINING_CONFIG["greater_is_better"],
        report_to=TRAINING_CONFIG["report_to"],
        logging_dir=os.path.join(LOGS_DIR, "tensorboard", run_name),
        dataloader_num_workers=TRAINING_CONFIG["dataloader_num_workers"],
        optim=TRAINING_CONFIG["optim"],
        neftune_noise_alpha=TRAINING_CONFIG["neftune_noise_alpha"],
        eval_strategy="steps",
        dataset_text_field="text",
        packing=False,
        max_length = TRAINING_CONFIF["max_length"]
    )

    callback = FinanceTrainingCallback(log_path=log_path)

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        args=training_args,
        callbacks=[callback],
    )

    logger.info("Starting training...")
    if train:
        trainer.train()
    logger.info("Finished training...")

    best_ckpt = trainer.state.best_model_checkpoint
    logger.info(f"Best checkpoint: {best_ckpt}")

    # Save adapter only (saves VRAM-safe)
    adapter_path = os.path.join(output_dir, "final_adapter")
    trainer.model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    logger.info(f"Adapter saved to: {adapter_path}")

    # Free memory
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return adapter_path


## Cell 9 — Evaluation Utilities


In [ ]:

bleu_metric  = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")


def _strip_output_tag(text: str) -> str:
    """Extract only the text inside <output>...</output>."""
    m = re.search(r"<output>(.*?)(?:</output>|$)", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()


def generate_predictions(
    model_key: str,
    adapter_path: str,
    test_samples: List[dict],
    batch_size: int = 8,
    max_new_tokens: int = 128,
) -> Tuple[List[str], List[str]]:
    """
    Load a trained LoRA adapter and generate predictions on test_samples.
    Returns (predictions, references).
    """
    model_cfg = MODEL_CONFIG[model_key]
    logger.info(f"Generating predictions: {adapter_path}")

    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"          # left-pad for generation

    base_model = AutoModelForCausalLM.from_pretrained(
        model_cfg["model_id"],
        quantization_config=BNBCONFIG,
        device_map={"": 0},
        dtype=torch.float32,
        attn_implementation="eager"
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    predictions, references = [], []

    for i in tqdm(range(0, len(test_samples), batch_size), desc="Generating"):
        batch = test_samples[i : i + batch_size]

        # Build inference prompts (output tag left open)
        prompts = []
        for s in batch:
            raw = s["text"]
            if "<output>" in raw:
                raw = raw[:raw.rfind("<output>") + len("<output>")]
            if model_key in ("smollm2", "qwen3"):
                p = f"<|im_start|>user\n{raw}<|im_end|>\n<|im_start|>assistant\n"
            elif model_key == "gemma3":
                p = f"<start_of_turn>user\n{raw}<end_of_turn>\n<start_of_turn>model\n"
            else:
                p = raw
            prompts.append(p)
            
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MODEL_CONFIG[model_key]["max_seq_length"],
        ).to(DEVICE)

        with torch.no_grad():
            out_ids = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            )
        
        # Decode only the newly generated tokens
        input_len = enc["input_ids"].shape[1]
        for j, ids in enumerate(out_ids):
            pred_text = tokenizer.decode(ids[input_len:], skip_special_tokens=True).strip()
            predictions.append(pred_text)
            references.append(str(batch[j].get("reference", "")))

    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, references


def compute_bleu(predictions: List[str], references: List[str]) -> float:
    refs_wrapped = [[r] for r in references]
    result = bleu_metric.compute(predictions=predictions, references=refs_wrapped)
    return round(result["score"], 4)


def compute_rouge(predictions: List[str], references: List[str]) -> dict:
    result = rouge_metric.compute(predictions=predictions, references=references)
    return {k: round(v, 4) for k, v in result.items()}


def compute_token_f1(prediction: str, reference: str) -> float:
    """Token-level F1 (standard QA metric)."""
    pred_tokens = prediction.lower().split()
    ref_tokens  = reference.lower().split()
    common = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens) if pred_tokens else 0.0
    recall    = len(common) / len(ref_tokens)  if ref_tokens  else 0.0
    if precision + recall == 0:
        return 0.0
    return round(2 * precision * recall / (precision + recall), 4)


def compute_sentiment_f1(predictions: List[str], references: List[str]) -> dict:
    """Macro + per-class F1 for sentiment classification."""
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    y_true, y_pred = [], []
    for p, r in zip(predictions, references):
        p_clean = p.lower().strip().split()[0] if p.strip() else "neutral"
        p_clean = p_clean.strip(".,;:")
        y_true.append(label_map.get(r.lower(), 1))
        y_pred.append(label_map.get(p_clean, 1))
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    report   = classification_report(y_true, y_pred,
                                     target_names=["negative", "neutral", "positive"],
                                     output_dict=True, zero_division=0)
    return {
        "macro_f1":    round(macro_f1, 4),
        "negative_f1": round(report["negative"]["f1-score"], 4),
        "neutral_f1":  round(report["neutral"]["f1-score"],  4),
        "positive_f1": round(report["positive"]["f1-score"], 4),
    }


def evaluate_run(
    model_key: str,
    adapter_path: str,
    test_samples: List[dict],
    task_group: str,
    run_label: str,
) -> dict:
    """
    Generate and evaluate predictions for a run.
    Returns a results dict with all applicable metrics.
    """
    predictions, references = generate_predictions(model_key, adapter_path, test_samples)

    results = {
        "run":        run_label,
        "model":      model_key,
        "task_group": task_group,
        "n_samples":  len(test_samples),
    }

    # BLEU — applicable to all generation tasks
    results["bleu"] = compute_bleu(predictions, references)

    # ROUGE — applicable to all generation tasks
    rouge = compute_rouge(predictions, references)
    results.update(rouge)

    # Token F1 — QA, NER, Summarization
    if task_group in ("qa", "ner", "summarization"):
        token_f1s = [compute_token_f1(p, r) for p, r in zip(predictions, references)]
        results["token_f1_mean"] = round(float(np.mean(token_f1s)), 4)

    # Classification F1 — Sentiment only
    if task_group == "sentiment":
        sent_f1 = compute_sentiment_f1(predictions, references)
        results.update(sent_f1)

    # Save predictions
    pred_path = os.path.join(RESULTS_DIR, f"{run_label}_predictions.jsonl")
    with open(pred_path, "w") as f:
        for p, r in zip(predictions, references):
            f.write(json.dumps({"prediction": p, "reference": r}) + "\n")

    logger.info(f"  Results for {run_label}: {results}")
    return results


## Cell 10 — Benchmark A: Merged Finetuning


In [ ]:

def run_merged_benchmark(train=True):
    """
    Benchmark A: Merge all dataset training splits, finetune each model once,
    evaluate each dataset's test split independently.
    """
    logger.info("\n" + "=" * 60)
    logger.info("BENCHMARK A — MERGED FINETUNING")
    logger.info("=" * 60)

    # Build merged train set (equal contribution — one pass per dataset)
    all_train, all_eval = [], []
    for key, split in DATASET_SPLITS.items():
        all_train.extend(split["train"])
        all_eval.extend(split["test"][:100])    # small eval slice for speed during training

    random.shuffle(all_train)
    random.shuffle(all_eval)
    logger.info(f"Merged train: {len(all_train)} | Merged eval: {len(all_eval)}")

    merged_results = []

    for model_key, model_cfg in MODEL_CONFIG.items():
        if not model_cfg["enabled"]:
            continue

        run_name   = f"merged_{model_key}"
        output_dir = os.path.join(BASE_OUTPUT_DIR, run_name)
        adapter_path = None
        
        adapter_path = run_training(
            model_key=model_key,
            run_name=run_name,
            train_samples=all_train,
            eval_samples=all_eval,
            output_dir=output_dir,
            train = train,
        )
        if adapter_path is None:
            continue

        # Evaluate per-dataset on its own test split
        for ds_key, split in DATASET_SPLITS.items():
            task_group = DATASET_CONFIG[ds_key]["task_group"]
            run_label  = f"merged_{model_key}_on_{ds_key}"
            res = evaluate_run(
                model_key=model_key,
                adapter_path=adapter_path,
                test_samples=split["test"],
                task_group=task_group,
                run_label=run_label,
            )
            res["benchmark"] = "merged"
            merged_results.append(res)

    # Persist
    out_path = os.path.join(RESULTS_DIR, "benchmark_A_merged.json")
    with open(out_path, "w") as f:
        json.dump(merged_results, f, indent=2)
    logger.info(f"Benchmark A results saved: {out_path}")
    return merged_results


## Cell 11 — Benchmark B: Individual Dataset Finetuning


In [ ]:

def run_individual_benchmark(train=True):
    """
    Benchmark B: For each (model, dataset) pair, finetune from scratch
    and evaluate on that dataset's own test split.
    """
    logger.info("\n" + "=" * 60)
    logger.info("BENCHMARK B — INDIVIDUAL DATASET FINETUNING")
    logger.info("=" * 60)

    individual_results = []

    for ds_key, split in DATASET_SPLITS.items():
        task_group   = DATASET_CONFIG[ds_key]["task_group"]
        train_samples = split["train"]
        test_samples  = split["test"]

        # Small eval slice from test set (used during training for early stopping)
        eval_slice = test_samples[:min(100, len(test_samples))]

        for model_key, model_cfg in MODEL_CONFIG.items():
            if not model_cfg["enabled"]:
                continue

            run_name   = f"indiv_{model_key}_{ds_key}"
            output_dir = os.path.join(BASE_OUTPUT_DIR, run_name)

            adapter_path = run_training(
                model_key=model_key,
                run_name=run_name,
                train_samples=train_samples,
                eval_samples=eval_slice,
                output_dir=output_dir,
                train = train,
            )
            if adapter_path is None:
                continue

            run_label = f"indiv_{model_key}_{ds_key}"
            res = evaluate_run(
                model_key=model_key,
                adapter_path=adapter_path,
                test_samples=test_samples,
                task_group=task_group,
                run_label=run_label,
            )
            res["benchmark"] = "individual"
            res["dataset"]   = ds_key
            individual_results.append(res)

    # Persist
    out_path = os.path.join(RESULTS_DIR, "benchmark_B_individual.json")
    with open(out_path, "w") as f:
        json.dump(individual_results, f, indent=2)
    logger.info(f"Benchmark B results saved: {out_path}")
    return individual_results


## Cell 12 — Results Aggregation & Summary Tables


In [ ]:

def build_summary_table(results: List[dict], label: str) -> pd.DataFrame:
    """Convert a list of result dicts to a readable DataFrame."""
    rows = []
    for r in results:
        row = {
            "Benchmark":  r.get("benchmark", label),
            "Model":      r.get("model", ""),
            "Dataset":    r.get("run", "").split("_on_")[-1] if "on_" in r.get("run","") else r.get("dataset",""),
            "Task":       r.get("task_group", ""),
            "N":          r.get("n_samples", ""),
            "BLEU":       r.get("bleu", ""),
            "ROUGE-1":    r.get("rouge1", ""),
            "ROUGE-2":    r.get("rouge2", ""),
            "ROUGE-L":    r.get("rougeL", ""),
            "Token-F1":   r.get("token_f1_mean", ""),
            "Macro-F1":   r.get("macro_f1", ""),
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    return df


def print_summary(merged_results_without_training:List[dict],merged_results: List[dict], individual: List[dict]):
    logger.info("\n" + "=" * 60)
    logger.info("FINAL RESULTS SUMMARY")
    logger.info("=" * 60)

    df_merged = build_summary_table(merged_results, "merged")
    df_merged_without_training  = build_summary_table(merged_results_without_training,"merged_no_train")
    # df_indiv  = build_summary_table(individual_results, "individual")
    df_all    = pd.concat([df_merged, df_merged_without_training], ignore_index=True)

    # Save
    csv_path = os.path.join(RESULTS_DIR, "all_results.csv")
    df_all.to_csv(csv_path, index=False)
    logger.info(f"Full results table: {csv_path}")

    # Print per-model summaries
    for model_key in MODEL_CONFIG:
        sub = df_all[df_all["Model"] == model_key]
        if sub.empty:
            continue
        logger.info(f"\n── {model_key.upper()} ──")
        print(sub[["Benchmark","Dataset","Task","BLEU","ROUGE-1","ROUGE-L",
                   "Token-F1","Macro-F1"]].to_string(index=False))

    # Per-task group averages
    logger.info("\n── Per-Task Averages (numeric columns) ──")
    numeric_cols = ["BLEU","ROUGE-1","ROUGE-2","ROUGE-L","Token-F1","Macro-F1"]
    for col in numeric_cols:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

    agg = (
        df_all.groupby(["Benchmark","Model","Task"])[numeric_cols]
        .mean()
        .round(4)
        .reset_index()
    )
    print(agg.to_string(index=False))

    agg_path = os.path.join(RESULTS_DIR, "aggregated_results.csv")
    agg.to_csv(agg_path, index=False)
    logger.info(f"Aggregated results: {agg_path}")

    return df_all, agg


## Cell 13 — Main Entry Point


In [ ]:
def main():
    logger.info("Financial LLM Finetuning Pipeline starting...")
    logger.info(f"Timestamp: {datetime.now().isoformat()}")
    logger.info(f"Enabled datasets : {[k for k,v in DATASET_CONFIG.items() if v.get('enabled')]}")
    logger.info(f"Enabled models   : {[k for k,v in MODEL_CONFIG.items()   if v.get('enabled')]}")

    # ── Benchmark A: Merged ──────────────────────────────────────────────────
    # merged_results_without_training = run_merged_benchmark(train=False)
    merged_results_with_training = run_merged_benchmark(train=True)

    # ── Benchmark B: Individual ──────────────────────────────────────────────
    # individual_results = run_individual_benchmark()

    # ── Summary ──────────────────────────────────────────────────────────────
    
    df_all, agg = print_summary([],merged_results_with_training,[])

    logger.info("\nAll done. Outputs written to /kaggle/working/results/")
    return df_all, agg


if __name__ == "__main__":
    try:
        df_all, agg = main()
    except Exception as e:
        logger.error(f"An error occurred during pipeline execution: {e}")
        import traceback
        traceback.print_exc()